# M14 on Kaggle: E3 and E4 on a partition variant, one seed

**Set two things in the first code cell:** `SEED` (42, 43 or 44) and `VARIANT`.

- `dpub_benign` (tier 1, seeds 42, 43 and 44): D_pub is the 5,000 benign rows only (D2's ablation).

It builds the variant's own partition (`splits/full/<timeline>/<VARIANT>`), then runs E3 (M7, without FedMD), M9's K sweep (`fixed_k10`, `fixed_k20`, `fixed_k30`) and `e4_added`, so K is chosen on the variant's own validation rows. Everything is written under `reports/mX/full/<VARIANT>/seed<SEED>/`; the headline reports are not touched.

It needs nothing from the other M14 notebooks, so it can run at the same time as them and as the M11 GPU session. Different seeds can run at the same time.

**Attach as input:** the private Kaggle dataset with the M1 `processed/` files.
**Settings:** Internet *On*. Accelerator: *None*.
**Run:** *Save Version → Save & Run All (Commit)*.

Each step runs in its own process and the zip is rewritten after each one. `exit code: 137` means the process was killed (usually out of memory).

**Output:** `/kaggle/working/m14_<VARIANT>_seed<SEED>_outputs.zip`. Download it and bring it back, also after a failure.


In [ ]:
SEED = 42                  # 42, 43 or 44
VARIANT = "dpub_benign"
VARIANTS = {"dpub_benign": "partition.variant=dpub_benign dpub.benign_only=true"}
TAG = f"{VARIANT}/seed{SEED}"
SET = f"--set resources.n_jobs=4 seed={SEED} {VARIANTS[VARIANT]}"   # 4 threads: results reproduce exactly only at the same thread count
import glob, os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert SEED in (42, 43, 44), SEED
assert "a variant (M14) reports into its own folder" in open("scripts/m2_build_partitions.py").read(), \
    "This clone predates the variant notebook. Restart the session so it clones the latest main."
assert glob.glob("/kaggle/input/**/data_manifest.json", recursive=True), "The processed dataset is not attached: Add Input -> your M1 processed dataset."


In [ ]:
# Build the variant's partition for the full pool. Its report goes to reports/m2/full/<VARIANT>, not over the headline one.
# The partition is always built at seed 42 (every seed shares it), so SEED is not passed here.
!python scripts/m2_build_partitions.py --env kaggle --pools full --set {VARIANTS[VARIANT]}; echo "exit code: $?"


In [ ]:
import shutil
REPS = [f"reports/m2/full/{VARIANT}", f"reports/m7/full/{TAG}", f"reports/m9/full/{TAG}"]
OUT = f"/kaggle/working/m14_{VARIANT}_seed{SEED}_outputs"
def pack():
    stage = "/kaggle/working/m14_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)
    for rep in REPS:
        if os.path.isdir(rep):
            shutil.copytree(rep, f"{stage}/{rep}", ignore=shutil.ignore_patterns("checkpoints", "seed4*"), dirs_exist_ok=True)
    for r in sorted(glob.glob("/kaggle/working/results/runs/*")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}", ignore=shutil.ignore_patterns("models"))
    shutil.make_archive(OUT, "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", OUT + ".zip")

def step(cmd):
    print("=====", cmd, flush=True)
    rc = subprocess.run(f"python -X faulthandler {cmd}", shell=True).returncode
    print("exit code:", rc, flush=True)
    pack()
    return rc


In [ ]:
ARGS = f"--env kaggle --pool full {SET} --tag {TAG}"
step(f"scripts/m7_distill.py {ARGS} --only e3 --resume")
step(f"scripts/m7_distill.py {ARGS} --report")          # M9's report reads E3 from it
for name in ["fixed_k10", "fixed_k20", "fixed_k30", "e4_added"]:
    step(f"scripts/m9_e4.py {ARGS} --only {name} --resume")
step(f"scripts/m9_e4.py {ARGS} --report")


In [ ]:
# What finished, and whether both runs have per-client counts
for m in ["m7", "m9"]:
    f = f"reports/{m}/full/{TAG}/confusions.csv"
    ok = os.path.exists(f) and "csv_client" in open(f).read()
    print(f"{m}: {'per-client counts present' if ok else 'MISSING (see the exit codes above)'}")
print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout[:3000])
